# qa_ticket ingestion

`tests/common/test_db_connection.py`의 PostgreSQL 접속 정보를 기준으로 `qa_ticket` 구조를 확인하고, `notebooks/navercafe.csv` 데이터를 적재합니다.

- 기본값은 `DRY_RUN = True`입니다. 미리보기 결과를 확인한 뒤 실제 적재 시 `False`로 변경하세요.
- `DB_PASSWORD`는 `.env` 또는 환경변수에서 읽습니다.
- CSV의 네이버 카페 작성자는 `community_users`에 `navercafe_<hash>@local.invalid` 형식의 식별 이메일로 매핑합니다.


In [ ]:
from __future__ import annotations

import csv
import hashlib
import json
import os
from datetime import datetime
from pathlib import Path
from typing import Any

import psycopg
from psycopg.rows import dict_row
from dotenv import load_dotenv

load_dotenv()

# Source: tests/common/test_db_connection.py
DB_HOST = os.environ.get("DB_HOST", "100.97.235.15")
DB_PORT = os.environ.get("DB_PORT", "5432")
DB_USER = os.environ.get("DB_USER", "game_cs_user")
DB_NAME = os.environ.get("DB_NAME", "game_cs")
DB_PASSWORD = os.environ.get("DB_PASSWORD")

CSV_PATH = Path("navercafe.csv")
if not CSV_PATH.exists():
    CSV_PATH = Path("notebooks/navercafe.csv")

DRY_RUN = True
LIMIT_ROWS: int | None = None  # 일부 테스트 적재 시 1000처럼 숫자로 변경
BATCH_SIZE = 500

if not DB_PASSWORD:
    raise RuntimeError("DB_PASSWORD 환경변수 또는 .env 값이 필요합니다.")

def connect() -> psycopg.Connection:
    return psycopg.connect(
        host=DB_HOST,
        port=DB_PORT,
        user=DB_USER,
        password=DB_PASSWORD,
        dbname=DB_NAME,
        connect_timeout=15,
    )

print(f"CSV: {CSV_PATH.resolve()}")
print(f"DB: {DB_USER}@{DB_HOST}:{DB_PORT}/{DB_NAME}")


## 1. DB 연결 및 테이블 구조 확인

In [ ]:
with connect() as conn:
    with conn.cursor(row_factory=dict_row) as cur:
        cur.execute("SELECT 1 AS ok")
        print("connection:", cur.fetchone()["ok"])

        cur.execute(
            """
            SELECT
                table_name,
                ordinal_position,
                column_name,
                data_type,
                is_nullable,
                column_default
            FROM information_schema.columns
            WHERE table_schema = 'public'
              AND table_name IN ('qa_ticket', 'community_users')
            ORDER BY table_name, ordinal_position
            """
        )
        columns = cur.fetchall()

        cur.execute("SELECT COUNT(*) AS cnt FROM qa_ticket")
        qa_ticket_count = cur.fetchone()["cnt"]

print("qa_ticket row count:", qa_ticket_count)
for row in columns:
    print(
        row["table_name"],
        row["ordinal_position"],
        row["column_name"],
        row["data_type"],
        "nullable=" + row["is_nullable"],
        "default=" + str(row["column_default"]),
    )


## 2. CSV 컬럼 및 적재 매핑 확인

In [ ]:
def read_csv_rows(limit: int | None = None) -> list[dict[str, str]]:
    rows: list[dict[str, str]] = []
    with CSV_PATH.open("r", encoding="utf-8-sig", newline="", errors="replace") as f:
        reader = csv.DictReader(f)
        for idx, row in enumerate(reader):
            if limit is not None and idx >= limit:
                break
            rows.append(row)
    return rows

sample_rows = read_csv_rows(3)
print("sample row count:", len(sample_rows))
print("columns:")
print(list(sample_rows[0].keys()) if sample_rows else [])

preview_fields = ["article_id", "title", "author", "created_at_dt", "article_url", "content"]
for row in sample_rows:
    print({field: row.get(field) for field in preview_fields})


In [ ]:
def parse_int(value: Any) -> int | None:
    if value is None or value == "":
        return None
    try:
        return int(str(value).strip())
    except ValueError:
        return None

def parse_datetime(value: Any) -> datetime | None:
    if value is None or value == "":
        return None
    text = str(value).strip()
    for fmt in ("%Y-%m-%d %H:%M:%S", "%Y-%m-%dT%H:%M:%S", "%Y.%m.%d. %H:%M"):
        try:
            return datetime.strptime(text, fmt)
        except ValueError:
            pass
    return None

def stable_author_email(row: dict[str, str]) -> str:
    raw_key = row.get("author_button_id") or row.get("author_profile_url") or row.get("author") or "unknown"
    cafe_id = row.get("cafe_id") or "unknown"
    digest = hashlib.sha1(f"{cafe_id}:{raw_key}".encode("utf-8", errors="replace")).hexdigest()[:16]
    return f"navercafe_{digest}@local.invalid"

def to_ticket(row: dict[str, str], user_id: int) -> dict[str, Any] | None:
    ticket_id = parse_int(row.get("article_id"))
    if ticket_id is None:
        return None

    content = row.get("content") or ""
    if not content and row.get("paragraphs_json"):
        try:
            content = "\n".join(json.loads(row["paragraphs_json"]))
        except json.JSONDecodeError:
            content = row.get("paragraphs_json", "")

    return {
        "ticket_id": ticket_id,
        "account_id": None,
        "user_id": user_id,
        "title": row.get("title") or "",
        "raw_query": content,
        "source_type": row.get("source") or "naver_cafe",
        "status": "open",
        "inquiry_created_at": parse_datetime(row.get("created_at_dt")) or parse_datetime(row.get("created_at")),
        "session_id": parse_int(row.get("cafe_id")),
    }


## 3. 적재 실행

`DRY_RUN = True`이면 DB 변경 없이 변환 결과만 확인합니다. 실제 적재하려면 첫 번째 코드 셀에서 `DRY_RUN = False`로 바꾸고 다시 실행하세요.

In [ ]:
def get_or_create_user(cur: psycopg.Cursor, row: dict[str, str], cache: dict[str, int]) -> int:
    email = stable_author_email(row)
    if email in cache:
        return cache[email]

    cur.execute("SELECT user_id FROM community_users WHERE email = %s", (email,))
    found = cur.fetchone()
    if found:
        user_id = int(found["user_id"] if isinstance(found, dict) else found[0])
        cache[email] = user_id
        return user_id

    cur.execute("SELECT COALESCE(MAX(user_id), 0) + 1 AS next_user_id FROM community_users")
    user_id = int(cur.fetchone()["next_user_id"])
    cur.execute(
        """
        INSERT INTO community_users (user_id, email, nickname, created_at, user_status, last_login_at)
        VALUES (%s, %s, %s, CURRENT_TIMESTAMP, %s, NULL)
        ON CONFLICT (user_id) DO NOTHING
        """,
        (user_id, email, row.get("author") or "naver_cafe_user", "active"),
    )
    cache[email] = user_id
    return user_id

def upsert_ticket(cur: psycopg.Cursor, ticket: dict[str, Any]) -> None:
    cur.execute(
        """
        INSERT INTO qa_ticket (
            ticket_id, account_id, user_id, title, raw_query,
            source_type, status, inquiry_created_at, session_id
        )
        VALUES (
            %(ticket_id)s, %(account_id)s, %(user_id)s, %(title)s, %(raw_query)s,
            %(source_type)s, %(status)s, %(inquiry_created_at)s, %(session_id)s
        )
        ON CONFLICT (ticket_id) DO UPDATE SET
            account_id = EXCLUDED.account_id,
            user_id = EXCLUDED.user_id,
            title = EXCLUDED.title,
            raw_query = EXCLUDED.raw_query,
            source_type = EXCLUDED.source_type,
            inquiry_created_at = EXCLUDED.inquiry_created_at,
            session_id = EXCLUDED.session_id
        """,
        ticket,
    )

rows = read_csv_rows(LIMIT_ROWS)
print(f"loaded csv rows: {len(rows):,}")

if DRY_RUN:
    fake_user_id = -1
    tickets = [to_ticket(row, fake_user_id) for row in rows[:5]]
    print("DRY_RUN=True: no database changes will be made")
    for ticket in tickets:
        print(ticket)
else:
    inserted_or_updated = 0
    skipped = 0
    user_cache: dict[str, int] = {}

    with connect() as conn:
        with conn.cursor(row_factory=dict_row) as cur:
            for start in range(0, len(rows), BATCH_SIZE):
                batch = rows[start:start + BATCH_SIZE]
                for row in batch:
                    user_id = get_or_create_user(cur, row, user_cache)
                    ticket = to_ticket(row, user_id)
                    if ticket is None:
                        skipped += 1
                        continue
                    upsert_ticket(cur, ticket)
                    inserted_or_updated += 1
                conn.commit()
                print(f"committed {min(start + BATCH_SIZE, len(rows)):,}/{len(rows):,}")

    print({"inserted_or_updated": inserted_or_updated, "skipped": skipped, "users": len(user_cache)})


## 4. 적재 결과 확인

In [ ]:
with connect() as conn:
    with conn.cursor(row_factory=dict_row) as cur:
        cur.execute("SELECT COUNT(*) AS cnt FROM qa_ticket WHERE source_type = %s", ("naver_cafe",))
        print("naver_cafe tickets:", cur.fetchone()["cnt"])

        cur.execute(
            """
            SELECT ticket_id, user_id, title, status, inquiry_created_at, session_id
            FROM qa_ticket
            WHERE source_type = %s
            ORDER BY inquiry_created_at DESC NULLS LAST, ticket_id DESC
            LIMIT 10
            """,
            ("naver_cafe",),
        )
        for row in cur.fetchall():
            print(dict(row))
